# 📚 Smart Library — Système de Recommandation
## Notebook 06 : KNN + ALS Collaborative Filtering avec Spark MLlib (Iceberg)

---

### Objectif
Construire un **système de recommandation de livres** basé sur les comportements
utilisateurs, en utilisant deux approches complémentaires.

### Approches implémentées
| Algorithme | Type | Description | Avantages |
|---|---|---|---|
| **KNN** | Filtrage collaboratif user-based | Trouve les utilisateurs similaires | Simple, interprétable |
| **ALS** | Matrix Factorization | Décompose la matrice user-item | Scalable, précis |

### Pipeline
```
Silver commandes + événements
         ↓
Matrice User-Item (utilisateur × livre)
         ↓
    ┌────┴────┐
   KNN       ALS
    └────┬────┘
         ↓
gold.recommandations (Iceberg)
```

---
## Étape 0 — Initialisation

In [1]:
import os, sys, time, re
from datetime import datetime
from pathlib import Path
from pyspark.sql.window  import Window

# ── Tuer toute session existante ─────────────────────────────
try:
    from pyspark.sql import SparkSession
    SparkSession.getActiveSession().stop()
except:
    pass

# ── Java ──────────────────────────────────────────────────────
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"]      = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

# ── Jars Iceberg + MinIO ──────────────────────────────────────
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.5.2,"
    "org.apache.iceberg:iceberg-aws-bundle:1.5.2,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.types import IntegerType, FloatType
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer

# ── Connexion MinIO + Lakekeeper ──────────────────────────────
MINIO_ENDPOINT       = "http://minio:9000"
MINIO_ACCESS         = "minioadmin"
MINIO_SECRET         = "minioadmin123"
CATALOG_NAME         = "lakehouse"
LAKEKEEPER_URI       = "http://lakekeeper:8181/catalog"
LAKEKEEPER_WAREHOUSE = "lakehouse"

NS_BRONZE = f"{CATALOG_NAME}.bronze"
NS_SILVER = f"{CATALOG_NAME}.silver"
NS_GOLD   = f"{CATALOG_NAME}.gold"

# ── Fonctions utilitaires Iceberg (équivalent hudi_options) ───
#
#  Avec Hudi : le type de table (COW/MOR) et la clé d'upsert se
#  déclarent au niveau de l'ÉCRITURE (options du DataFrameWriter).
#  Avec Iceberg : ce sont des PROPRIÉTÉS DE TABLE, et l'UPSERT
#  s'exprime explicitement en SQL via MERGE INTO.
#
def iceberg_table_properties(type_table="copy-on-write"):
    """
    type_table :
      - "copy-on-write" → équivalent Hudi COW
      - "merge-on-read"  → équivalent Hudi MOR
    """
    return {
        "format-version"    : "2",
        "write.delete.mode" : type_table,
        "write.update.mode" : type_table,
        "write.merge.mode"  : type_table,
    }

def create_or_replace_table(df, table_name, partition_cols=None, type_table="copy-on-write"):
    """Équivalent de : df.write.format('hudi')....mode('overwrite').save(chemin)"""
    # Iceberg (ClusteredWriter) exige que les lignes soient triées/regroupées
    # par colonne de partition AVANT l'écriture — contrairement à Hudi qui
    # gère le clustering en interne. Sans ce tri : "Incoming records violate
    # the writer assumption that records are clustered by spec and partition".
    if partition_cols:
        df = df.sort(*partition_cols)
    writer = df.writeTo(table_name)
    if partition_cols:
        writer = writer.partitionedBy(*[F.col(c) for c in partition_cols])
    for k, v in iceberg_table_properties(type_table).items():
        writer = writer.tableProperty(k, v)
    writer.createOrReplace()

def append_table(df, table_name):
    """Ajout de lignes (insert simple, pas d'upsert)."""
    df.writeTo(table_name).append()

def upsert_table(spark, df, table_name, key_cols):
    """UPSERT via MERGE INTO — équivalent fonctionnel de l'UPSERT automatique Hudi."""
    vue = f"_staging_{table_name.split('.')[-1]}"
    df.createOrReplaceTempView(vue)
    condition = " AND ".join([f"t.{c} = s.{c}" for c in key_cols])
    spark.sql(f"""
        MERGE INTO {table_name} t
        USING {vue} s
        ON {condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

def snapshots(spark, table_name):
    """Retourne l'historique des snapshots Iceberg (équivalent de la timeline Hudi)."""
    return spark.sql(f"SELECT * FROM {table_name}.snapshots ORDER BY committed_at")

# ── SparkSession ──────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("SmartLibrary-Recommandations")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", LAKEKEEPER_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", LAKEKEEPER_WAREHOUSE)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint",           MINIO_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.access-key-id",     MINIO_ACCESS)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.secret-access-key", MINIO_SECRET)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .config("spark.serializer",
            "org.apache.spark.serializer.KryoSerializer")
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ── Création des namespaces Medallion (idempotent) ────────────
for ns in (NS_BRONZE, NS_SILVER, NS_GOLD):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {ns}")

print(f"✅ SparkSession démarrée — version {spark.version}")
print(f"   Catalogue Iceberg : {CATALOG_NAME} (REST → Lakekeeper → Postgres)")
print(f"   Stockage          : {MINIO_ENDPOINT} (bucket lakehouse)")

SILVER_COMMANDES = f"{NS_SILVER}.commandes"
SILVER_LIVRES    = f"{NS_SILVER}.livres"
GOLD_RECOMM      = f"{NS_GOLD}.recommandations"

✅ SparkSession démarrée — version 3.4.0
   Catalogue Iceberg : lakehouse (REST → Lakekeeper → Postgres)
   Stockage          : http://minio:9000 (bucket lakehouse)


---
## Étape 1 — Construction de la matrice User-Item

In [2]:
# ── Charger les interactions utilisateur-livre ────────────────
df_commandes = (
    spark.table(SILVER_COMMANDES)
    .filter(F.col("statut").isin(["livré", "en_cours"]))
    .select("utilisateur_id", "livre_id", "quantite", "montant_total",
            "genre", "titre")
)

df_livres = (
    spark.table(SILVER_LIVRES)
    .select("livre_id", "titre", "genre", "auteur",
            "note_moyenne", "score_popularite")
)

print(f"📊 Interactions chargées : {df_commandes.count():,}")
print(f"   Utilisateurs uniques  : {df_commandes.select('utilisateur_id').distinct().count():,}")
print(f"   Livres uniques        : {df_commandes.select('livre_id').distinct().count():,}")

📊 Interactions chargées : 4,265
   Utilisateurs uniques  : 981
   Livres uniques        : 500


In [3]:
# ── Construction du score d'interaction ───────────────────────
# Score = nb achats × 5 + nb genres distincts × 2
# Normalisation entre 0 et 5 pour l'ALS

df_interactions = (
    df_commandes
    .groupBy("utilisateur_id", "livre_id")
    .agg(
        F.sum("quantite").alias("nb_achats"),
        F.sum("montant_total").alias("montant_total"),
        F.first("genre").alias("genre"),
        F.first("titre").alias("titre")
    )
    # Score normalisé entre 1 et 5
    .withColumn("score_brut",
                F.col("nb_achats").cast("double"))
)

# Normalisation min-max vers [1, 5]
stats = df_interactions.agg(
    F.min("score_brut").alias("min_score"),
    F.max("score_brut").alias("max_score")
).first()

min_s = float(stats["min_score"])
max_s = float(stats["max_score"])

df_interactions = (
    df_interactions
    .withColumn("rating",
                F.round(
                    1 + (F.col("score_brut") - min_s) /
                    (max_s - min_s + 0.001) * 4, 2
                ).cast(FloatType()))
)

print(f"✅ Matrice User-Item construite")
print(f"   Paires (user, livre) : {df_interactions.count():,}")
print()
df_interactions.select(
    "utilisateur_id", "livre_id", "titre",
    "nb_achats", "rating"
).show(8, truncate=True)

✅ Matrice User-Item construite
   Paires (user, livre) : 4,248

+--------------+--------+--------------------+---------+------+
|utilisateur_id|livre_id|               titre|nb_achats|rating|
+--------------+--------+--------------------+---------+------+
|      USR00001|  BK0034|Chaîne Acte Carac...|        3|  1.89|
|      USR00001|  BK0154|    Police Expliquer|        5|  2.78|
|      USR00001|  BK0267|                Même|        4|  2.33|
|      USR00001|  BK0364|Quarante Être Fen...|        3|  1.89|
|      USR00001|  BK0416|Plein Joie Pousse...|        1|   1.0|
|      USR00001|  BK0468|    Éclater Discours|        1|   1.0|
|      USR00002|  BK0206|Rôle Barbe Échapp...|        2|  1.44|
|      USR00002|  BK0389|          Pays Obéir|        5|  2.78|
+--------------+--------+--------------------+---------+------+
only showing top 8 rows



---
## Étape 2 — Approche 1 : KNN (K-Nearest Neighbors)
### Filtrage collaboratif basé sur les utilisateurs similaires

In [4]:
# ── Profil utilisateur : genres préférés ──────────────────────
df_profil_user = (
    df_commandes
    .groupBy("utilisateur_id", "genre")
    .agg(F.sum("quantite").alias("nb_achats_genre"))
)

# Top genre par utilisateur
window_user = Window.partitionBy("utilisateur_id").orderBy(
    F.col("nb_achats_genre").desc()
)

df_genre_prefere = (
    df_profil_user
    .withColumn("rang", F.rank().over(window_user))
    .filter(F.col("rang") == 1)
    .select("utilisateur_id",
            F.col("genre").alias("genre_prefere"),
            F.col("nb_achats_genre").alias("achats_genre_prefere"))
)

print(f"✅ Profils utilisateurs : {df_genre_prefere.count():,}")
print()
print("Distribution des genres préférés :")
df_genre_prefere.groupBy("genre_prefere").count() \
    .orderBy("count", ascending=False).show(10)

✅ Profils utilisateurs : 1,234

Distribution des genres préférés :
+--------------------+-----+
|       genre_prefere|count|
+--------------------+-----+
|            Jeunesse|  141|
|Développement per...|  137|
|            Policier|  134|
|          Biographie|  133|
|               Roman|  125|
|         Philosophie|  118|
|            Sciences|  116|
|              Poésie|  116|
|     Science-fiction|  114|
|            Histoire|  100|
+--------------------+-----+



In [5]:
# ── KNN : recommandations basées sur le genre préféré ─────────
# Pour chaque utilisateur :
# 1. Identifier son genre préféré
# 2. Trouver les livres populaires de ce genre qu'il n'a pas encore achetés

# Livres déjà achetés par chaque utilisateur
df_deja_achetes = (
    df_commandes
    .select("utilisateur_id", "livre_id")
    .distinct()
)

# Top livres par genre (basé sur score popularité)
df_top_genre = (
    df_livres
    .withColumn("rang_genre",
                F.rank().over(
                    Window.partitionBy("genre")
                          .orderBy(F.col("score_popularite").desc())
                ))
    .filter(F.col("rang_genre") <= 10)  # Top 10 par genre
)

# Jointure : utilisateur × top livres de son genre préféré
df_reco_knn = (
    df_genre_prefere
    .join(
        df_top_genre.select(
            F.col("genre").alias("genre_prefere"),
            "livre_id", "titre", "auteur",
            "note_moyenne", "score_popularite", "rang_genre"
        ),
        on="genre_prefere",
        how="inner"
    )
    # Exclure les livres déjà achetés
    .join(
        df_deja_achetes,
        on=["utilisateur_id", "livre_id"],
        how="left_anti"   # Garder uniquement ceux NON achetés
    )
    .withColumn("methode",       F.lit("KNN"))
    .withColumn("score_reco",
                F.round(F.col("score_popularite"), 3))
    .withColumn("rang_recommandation",
                F.rank().over(
                    Window.partitionBy("utilisateur_id")
                          .orderBy(F.col("score_popularite").desc())
                ))
    .filter(F.col("rang_recommandation") <= 5)  # Top 5 par user
)

n_knn = df_reco_knn.count()
n_users_knn = df_reco_knn.select("utilisateur_id").distinct().count()
print(f"✅ Recommandations KNN générées")
print(f"   {n_knn:,} recommandations pour {n_users_knn:,} utilisateurs")
print()
print("Exemple — top 5 recommandations pour un utilisateur :")
premier_user = df_reco_knn.select("utilisateur_id").first()["utilisateur_id"]
df_reco_knn.filter(F.col("utilisateur_id") == premier_user) \
    .select("utilisateur_id", "genre_prefere", "rang_recommandation",
            "titre", "note_moyenne", "score_reco") \
    .orderBy("rang_recommandation") \
    .show(5, truncate=True)

✅ Recommandations KNN générées
   4,905 recommandations pour 981 utilisateurs

Exemple — top 5 recommandations pour un utilisateur :
+--------------+--------------------+-------------------+--------------------+------------+----------+
|utilisateur_id|       genre_prefere|rang_recommandation|               titre|note_moyenne|score_reco|
+--------------+--------------------+-------------------+--------------------+------------+----------+
|      USR00001|Développement per...|                  1|        Secret Parti|         5.0|     30.55|
|      USR00001|            Sciences|                  2|        Oncle Contre|         4.8|     29.09|
|      USR00001|Développement per...|                  3|Simple Lumière Hiver|         4.8|     28.21|
|      USR00001|Développement per...|                  4|Trait Officier Fa...|         4.5|     27.73|
|      USR00001|            Sciences|                  5|     Grâce Guère Nez|         4.5|     26.12|
+--------------+--------------------+------

---
## Étape 3 — Approche 2 : ALS (Alternating Least Squares)
### Matrix Factorization — méthode scalable recommandée pour la production

In [6]:
# ── Vérification SparkSession active ─────────────────────────
from pyspark.sql import SparkSession
from pyspark.context import SparkContext

sc = SparkContext._active_spark_context
if sc is None:
    print("⚠️ SparkContext perdu — réinitialisation...")
    spark = (
        SparkSession.builder
        .appName("SmartLibrary-Recommandations")
        .master("local[2]")
        .config("spark.driver.memory", "2g")
        .config("spark.sql.extensions",
                "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
        .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
        .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", LAKEKEEPER_URI)
        .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", LAKEKEEPER_WAREHOUSE)
        .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
                "org.apache.iceberg.aws.s3.S3FileIO")
        .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint",           MINIO_ENDPOINT)
        .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.access-key-id",     MINIO_ACCESS)
        .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.secret-access-key", MINIO_SECRET)
        .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.path-style-access", "true")
        .config("spark.sql.defaultCatalog", CATALOG_NAME)
        .config("spark.serializer",
                "org.apache.spark.serializer.KryoSerializer")
        .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
        .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS)
        .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.impl",
                "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.sql.shuffle.partitions", "4")
        .getOrCreate()
    )
    print("✅ SparkSession réinitialisée")
else:
    print(f"✅ SparkContext actif — version {spark.version}")

# ── Encodage numérique des IDs (requis par ALS) ───────────────
print("=" * 55)
print("  ALS — Matrix Factorization")
print("=" * 55)

indexer_user  = StringIndexer(inputCol="utilisateur_id", outputCol="user_idx")
indexer_livre = StringIndexer(inputCol="livre_id",       outputCol="livre_idx")

model_user  = indexer_user.fit(df_interactions)
model_livre = indexer_livre.fit(df_interactions)

# ✅ Syntaxe correcte PySpark : model.transform(df)
df_encoded = model_user.transform(df_interactions)
df_als_input = (
    model_livre.transform(df_encoded)
    .select(
        F.col("user_idx").cast(IntegerType()),
        F.col("livre_idx").cast(IntegerType()),
        F.col("rating")
    )
)

print(f"✅ Matrice encodée : {df_als_input.count():,} interactions")
print(f"   Users encodés  : {df_als_input.select('user_idx').distinct().count():,}")
print(f"   Livres encodés : {df_als_input.select('livre_idx').distinct().count():,}")
df_als_input.show(5)

✅ SparkContext actif — version 3.4.0
  ALS — Matrix Factorization
✅ Matrice encodée : 4,248 interactions
   Users encodés  : 981
   Livres encodés : 500
+--------+---------+------+
|user_idx|livre_idx|rating|
+--------+---------+------+
|     769|      477|  1.44|
|     768|      128|  2.33|
|     175|      380|  1.44|
|     394|       14|   1.0|
|     235|      101|  1.89|
+--------+---------+------+
only showing top 5 rows



In [7]:
# ── Entraînement ALS ──────────────────────────────────────────
# Split train / test
df_train, df_test = df_als_input.randomSplit([0.8, 0.2], seed=42)

print(f"  Train : {df_train.count():,} interactions")
print(f"  Test  : {df_test.count():,} interactions")

# Modèle ALS
als = ALS(
    maxIter       = 10,
    regParam      = 0.1,
    rank          = 10,          # Nb de facteurs latents
    userCol       = "user_idx",
    itemCol       = "livre_idx",
    ratingCol     = "rating",
    coldStartStrategy = "drop"   # Ignore les users/items inconnus
)

debut = time.time()
model_als = als.fit(df_train)
temps_train = round(time.time() - debut, 2)

print(f"\n✅ Modèle ALS entraîné en {temps_train}s")
print(f"   Rang (facteurs latents) : {als.getRank()}")
print(f"   Iterations max          : {als.getMaxIter()}")
print(f"   Régularisation          : {als.getRegParam()}")

  Train : 3,450 interactions
  Test  : 798 interactions

✅ Modèle ALS entraîné en 6.49s
   Rang (facteurs latents) : 10
   Iterations max          : 10
   Régularisation          : 0.1


In [8]:
# ── Évaluation du modèle ──────────────────────────────────────
predictions = model_als.transform(df_test)

evaluator = RegressionEvaluator(
    metricName  = "rmse",
    labelCol    = "rating",
    predictionCol = "prediction"
)

rmse = evaluator.evaluate(predictions)

evaluator_mae = RegressionEvaluator(
    metricName  = "mae",
    labelCol    = "rating",
    predictionCol = "prediction"
)
mae = evaluator_mae.evaluate(predictions)

print("=" * 45)
print("  ÉVALUATION DU MODÈLE ALS")
print("=" * 45)
print(f"  RMSE (Root Mean Square Error) : {round(rmse, 4)}")
print(f"  MAE  (Mean Absolute Error)    : {round(mae, 4)}")
print()
print("  Interprétation :")
print(f"  Le modèle prédit les notes avec une erreur")
print(f"  moyenne de {round(mae, 2)} point sur une échelle de 1 à 5")

print("\n  Exemples de prédictions :")
predictions.select("user_idx", "livre_idx", "rating", "prediction") \
    .orderBy(F.col("rating").desc()) \
    .show(8)

  ÉVALUATION DU MODÈLE ALS
  RMSE (Root Mean Square Error) : 1.0782
  MAE  (Mean Absolute Error)    : 0.881

  Interprétation :
  Le modèle prédit les notes avec une erreur
  moyenne de 0.88 point sur une échelle de 1 à 5

  Exemples de prédictions :
+--------+---------+------+----------+
|user_idx|livre_idx|rating|prediction|
+--------+---------+------+----------+
|      73|       84|  2.78| 1.6005063|
|     254|      294|  2.78| 1.7715182|
|     109|      165|  2.78| 1.7025805|
|      50|        0|  2.78| 1.4083768|
|     150|      172|  2.78|  1.333776|
|      70|       33|  2.78|  1.195651|
|     150|      177|  2.78| 1.7815546|
|     171|      134|  2.78| 1.7638935|
+--------+---------+------+----------+
only showing top 8 rows



In [9]:
# ── Génération des recommandations ALS ───────────────────────
# Top 5 recommandations par utilisateur
df_reco_als_raw = model_als.recommendForAllUsers(5)

# Aplatir la colonne recommendations (array of struct)
df_reco_als_flat = (
    df_reco_als_raw
    .select(
        F.col("user_idx"),
        F.posexplode(F.col("recommendations")).alias("rang", "rec")
    )
    .select(
        F.col("user_idx"),
        (F.col("rang") + 1).alias("rang_recommandation"),
        F.col("rec.livre_idx").alias("livre_idx"),
        F.col("rec.rating").alias("score_als")
    )
)

# Table de correspondance idx → ID original
df_user_map  = (
    model_user.transform(df_interactions)
    .select("utilisateur_id",
            F.col("user_idx").cast(IntegerType()))
    .distinct()
)
df_livre_map = (
    model_livre.transform(df_interactions)
    .select("livre_id",
            F.col("livre_idx").cast(IntegerType()))
    .distinct()
)

# Rejoindre avec les IDs originaux et les titres
df_reco_als = (
    df_reco_als_flat
    .join(df_user_map,  on="user_idx",  how="inner")
    .join(df_livre_map, on="livre_idx", how="inner")
    .join(
        df_livres.select("livre_id", "titre", "genre", "auteur", "note_moyenne"),
        on="livre_id",
        how="left"
    )
    .withColumn("methode",    F.lit("ALS"))
    .withColumn("score_reco", F.round(F.col("score_als"), 3))
)

n_als = df_reco_als.count()
n_users_als = df_reco_als.select("utilisateur_id").distinct().count()
print(f"✅ Recommandations ALS générées")
print(f"   {n_als:,} recommandations pour {n_users_als:,} utilisateurs")
print()
print("Exemple — top 5 ALS pour un utilisateur :")
premier_user_als = df_reco_als.select("utilisateur_id").first()["utilisateur_id"]
df_reco_als.filter(F.col("utilisateur_id") == premier_user_als) \
    .select("utilisateur_id", "rang_recommandation",
            "titre", "genre", "note_moyenne", "score_reco") \
    .orderBy("rang_recommandation") \
    .show(5, truncate=True)

✅ Recommandations ALS générées
   4,815 recommandations pour 963 utilisateurs

Exemple — top 5 ALS pour un utilisateur :
+--------------+-------------------+--------------------+--------------------+------------+----------+
|utilisateur_id|rang_recommandation|               titre|               genre|note_moyenne|score_reco|
+--------------+-------------------+--------------------+--------------------+------------+----------+
|      USR00214|                  1|    Pauvre Remplacer|          Biographie|         4.0|     2.842|
|      USR00214|                  2|Saint Religion At...|            Policier|         2.2|     2.653|
|      USR00214|                  3|Trait Officier Fa...|Développement per...|         4.5|     2.638|
|      USR00214|                  4|Grandir Note Prom...|            Jeunesse|         4.6|     2.398|
|      USR00214|                  5|  Poser Terre Social|               Roman|         4.9|     2.355|
+--------------+-------------------+-------------------

---
## Étape 4 — Fusion KNN + ALS → Table Gold Recommandations

In [10]:
# ── Colonnes communes aux deux méthodes ───────────────────────

# KNN a "genre_prefere" au lieu de "genre" → renommer
df_knn_final = (
    df_reco_knn
    .withColumnRenamed("genre_prefere", "genre")
    .select(
        "utilisateur_id", "livre_id", "titre", "genre", "auteur",
        "note_moyenne", "rang_recommandation", "methode", "score_reco"
    )
)

# ALS a déjà "genre"
df_als_final = df_reco_als.select(
    "utilisateur_id", "livre_id", "titre", "genre", "auteur",
    "note_moyenne", "rang_recommandation", "methode", "score_reco"
)

# Union des deux méthodes
df_recommandations = (
    df_knn_final.union(df_als_final)
    .withColumn("recommandation_id",
                F.concat_ws("_",
                    F.col("utilisateur_id"),
                    F.col("livre_id"),
                    F.col("methode")))
    .withColumn("updated_at",
                F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_methode", F.col("methode"))
)

n_total = df_recommandations.count()
print(f"✅ Table Gold Recommandations")
print(f"   Total         : {n_total:,} recommandations")
print(f"   KNN           : {df_knn_final.count():,}")
print(f"   ALS           : {df_als_final.count():,}")

df_recommandations.groupBy("methode").agg(
    F.count("*").alias("nb_reco"),
    F.countDistinct("utilisateur_id").alias("nb_users"),
    F.round(F.avg("score_reco"), 3).alias("score_moyen")
).show()

✅ Table Gold Recommandations
   Total         : 9,720 recommandations
   KNN           : 4,905
   ALS           : 4,815
+-------+-------+--------+-----------+
|methode|nb_reco|nb_users|score_moyen|
+-------+-------+--------+-----------+
|    KNN|   4905|     981|     27.292|
|    ALS|   4815|     963|      2.172|
+-------+-------+--------+-----------+



In [11]:
# ── Écriture Iceberg Gold ──────────────────────────────────────
create_or_replace_table(
    df_recommandations, GOLD_RECOMM,
    partition_cols=["partition_methode"], type_table="copy-on-write"
)

n = spark.table(GOLD_RECOMM).count()
print(f"✅ {GOLD_RECOMM} — {n:,} recommandations écrites dans Iceberg")

✅ lakehouse.gold.recommandations — 9,720 recommandations écrites dans Iceberg


---
## Étape 5 — Comparaison KNN vs ALS

In [12]:
# ── Analyse comparative ───────────────────────────────────────
print("=" * 62)
print("  COMPARAISON KNN vs ALS")
print("=" * 62)

# Genres recommandés par chaque méthode
print("\n  Distribution des genres recommandés :")
df_recommandations.groupBy("methode", "genre").count() \
    .orderBy("methode", F.col("count").desc()) \
    .show(20, truncate=False)

# Utilisateurs couverts par les deux méthodes
users_knn = set([r["utilisateur_id"] for r in df_knn_final.select("utilisateur_id").distinct().collect()])
users_als = set([r["utilisateur_id"] for r in df_als_final.select("utilisateur_id").distinct().collect()])

print(f"  Utilisateurs couverts KNN uniquement : {len(users_knn - users_als):,}")
print(f"  Utilisateurs couverts ALS uniquement : {len(users_als - users_knn):,}")
print(f"  Utilisateurs couverts par les DEUX   : {len(users_knn & users_als):,}")
print()
print("  AVANTAGES")
print("  KNN :")
print("    ✅ Interprétable — on sait pourquoi on recommande")
print("    ✅ Pas besoin d'entraînement — calcul à la demande")
print("    ❌ Limité aux genres connus — pas de découverte")
print()
print("  ALS :")
print("    ✅ Découverte — peut recommander des genres inattendus")
print("    ✅ Scalable — Spark MLlib distribué")
print(f"   ✅ Précis — RMSE = {round(rmse, 3)} sur échelle 1-5")
print("    ❌ Boîte noire — difficile à expliquer")

  COMPARAISON KNN vs ALS

  Distribution des genres recommandés :
+-------+-----------------------+-----+
|methode|genre                  |count|
+-------+-----------------------+-----+
|ALS    |Jeunesse               |641  |
|ALS    |Biographie             |608  |
|ALS    |Histoire               |563  |
|ALS    |Sciences               |551  |
|ALS    |Science-fiction        |499  |
|ALS    |Développement personnel|443  |
|ALS    |Policier               |439  |
|ALS    |Philosophie            |435  |
|ALS    |Poésie                 |365  |
|ALS    |Roman                  |271  |
|KNN    |Policier               |553  |
|KNN    |Biographie             |548  |
|KNN    |Jeunesse               |545  |
|KNN    |Développement personnel|543  |
|KNN    |Poésie                 |515  |
|KNN    |Philosophie            |504  |
|KNN    |Roman                  |497  |
|KNN    |Science-fiction        |452  |
|KNN    |Sciences               |393  |
|KNN    |Histoire               |355  |
+-------+-----

---
## Étape 6 — Bilan final du projet

In [13]:
# ── Bilan complet de toutes les tables ────────────────────────
print("=" * 65)
print("  BILAN FINAL — SMART LIBRARY DATA LAKEHOUSE (ICEBERG)")
print("=" * 65)

toutes_tables = [
    # Bronze
    ("Bronze", "bronze.utilisateurs",       f"{NS_BRONZE}.utilisateurs"),
    ("Bronze", "bronze.livres",             f"{NS_BRONZE}.livres"),
    ("Bronze", "bronze.commandes",          f"{NS_BRONZE}.commandes"),
    ("Bronze", "bronze.stock_cow",          f"{NS_BRONZE}.stock_cow"),
    ("Bronze", "bronze.stock_mor",          f"{NS_BRONZE}.stock_mor"),
    ("Bronze", "bronze.catalogue",          f"{NS_BRONZE}.catalogue"),
    ("Bronze", "bronze.factures",           f"{NS_BRONZE}.factures"),
    # Silver
    ("Silver", "silver.utilisateurs",       f"{NS_SILVER}.utilisateurs"),
    ("Silver", "silver.livres",             f"{NS_SILVER}.livres"),
    ("Silver", "silver.commandes",          f"{NS_SILVER}.commandes"),
    ("Silver", "silver.stock",              f"{NS_SILVER}.stock"),
    ("Silver", "silver.catalogue",          f"{NS_SILVER}.catalogue"),
    # Gold
    ("Gold",   "gold.top_livres",           f"{NS_GOLD}.top_livres"),
    ("Gold",   "gold.ventes_periode",       f"{NS_GOLD}.ventes_periode"),
    ("Gold",   "gold.comportement_utilisateurs", f"{NS_GOLD}.comportement_utilisateurs"),
    ("Gold",   "gold.historique_ventes",    f"{NS_GOLD}.historique_ventes"),
    ("Gold",   "gold.tendances",            f"{NS_GOLD}.tendances"),
    ("Gold",   "gold.tendances_temps_reel", f"{NS_GOLD}.tendances_temps_reel"),
    ("Gold",   "gold.recommandations",      f"{NS_GOLD}.recommandations"),
]

couche_actuelle = ""
total_global = 0

for couche, nom, table in toutes_tables:
    if couche != couche_actuelle:
        print(f"\n  ── {couche} ──")
        couche_actuelle = couche
    try:
        n = spark.table(table).count()
        total_global += n
        print(f"  {nom:<36} {n:>8,}  ✅")
    except:
        print(f"  {nom:<36} {'N/A':>8}  ⚠️")

print()
print("=" * 65)
print(f"  TOTAL GLOBAL : {total_global:,} lignes dans le Lakehouse")
print("=" * 65)
print()
print("  Fonctionnalités Iceberg démontrées :")
print("  ✅ copy-on-write & merge-on-read — deux modes d'écriture comparés")
print("  ✅ UPSERT (MERGE INTO) — mises à jour ciblées sans réécriture")
print("  ✅ Time Travel      — lecture à n'importe quel snapshot passé")
print("  ✅ Changelog        — lecture des seuls changements")
print("  ✅ Schema evolution — colonnes ajoutées sans casser les tables")
print("  ✅ Audit trail      — traçabilité complète via {table}.snapshots")
print("  ✅ Catalogue REST   — Lakekeeper + PostgreSQL, indépendant du moteur de calcul")
print()
print("  Valeur business démontrée :")
print("  ✅ Top livres       — bestsellers avec score de tendance")
print("  ✅ Analyse RFM      — segmentation Champion/Fidèle/À risque")
print("  ✅ Batch+Streaming  — score unifié 60/40")
print("  ✅ Recommandations  — KNN (interprétable) + ALS (scalable)")
print(f"  ✅ ALS RMSE         — {round(rmse, 3)} sur échelle 1-5")
print()
print("  🎓 Projet Smart Library Data Lakehouse (Apache Iceberg) — COMPLET")

  BILAN FINAL — SMART LIBRARY DATA LAKEHOUSE (ICEBERG)

  ── Bronze ──
  bronze.utilisateurs                     1,000  ✅
  bronze.livres                             500  ✅
  bronze.commandes                        5,000  ✅
  bronze.stock_cow                          500  ✅
  bronze.stock_mor                          500  ✅
  bronze.catalogue                          304  ✅
  bronze.factures                            50  ✅

  ── Silver ──
  silver.utilisateurs                     1,000  ✅
  silver.livres                             500  ✅
  silver.commandes                        5,000  ✅
  silver.stock                              530  ✅
  silver.catalogue                          291  ✅

  ── Gold ──
  gold.top_livres                           500  ✅
  gold.ventes_periode                       955  ✅
  gold.comportement_utilisateurs            993  ✅
  gold.historique_ventes                  4,003  ✅
  gold.tendances                              5  ✅
  gold.tendances_temps_reel     

In [14]:
spark.stop()
print("✅ SparkSession fermée")
print("🎓 Tous les notebooks sont exécutés avec succès !")
print("   MinIO Console  → http://localhost:9001")
print("   Lakekeeper API → http://localhost:8181")
print("   JupyterLab     → http://localhost:8888")

✅ SparkSession fermée
🎓 Tous les notebooks sont exécutés avec succès !
   MinIO Console  → http://localhost:9001
   Lakekeeper API → http://localhost:8181
   JupyterLab     → http://localhost:8888
